<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/Day27_Structured%20Tool%20Calling%20with%20the%20OpenAI%20Functions%20API/Structured_Tool_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 27 — Structured Tool Calling with the OpenAI Functions API

**Focus Area:** Structured Tool Calling

Day 26 built a manual ReAct agent that parsed tool calls out of raw LLM text
(`Action: tool_name(args)`) using regular expressions. That's fragile: any
deviation in formatting — missing quotes, different brackets, a
conversational aside before the `Action:` line — breaks the parser.

This notebook rebuilds the same agent using **OpenAI's function calling
API**, where the model returns a structured `tool_calls` field (guaranteed
valid JSON) instead of free text, and measures the reliability difference
directly against the Day 26 approach.

**Contents**
1. Tool implementations (`search_documents`, `get_weather_stub`, `calculate`, `get_today`)
2. Day 26 recap — manual text-parsing agent (+ 3 formatting-drift failures)
3. Day 27 — JSON function schemas + structured tool-calling loop
4. Five-problem comparison: Day 26 vs Day 27
5. Two-sequential-tool-call trace (search → calculate)
6. Argument validation wrapper + 3 bad-argument cases it catches
7. FastAPI `POST /agent` endpoint
8. Reliability comparison document

> **Note on running this notebook:** to make every cell runnable end-to-end
> without requiring your own OpenAI API key or network access, this
> notebook includes an `OpenAIClientAdapter` that uses the **real**
> `openai` Python client when `OPENAI_API_KEY` is set in the environment,
> and otherwise falls back to a small **deterministic mock client** that
> mimics the exact response shape of `chat.completions.create(..., tools=...)`
> (`choices[0].message.tool_calls[i].function.{name, arguments}`, with
> `arguments` as a JSON string, exactly like the real API). Swap in your key
> and re-run to see it hit the live API with no code changes.


## 1. Tool implementations

Four plain Python functions — these are the *actual* functions that get executed once a tool call is parsed/validated.

In [1]:
import json, re, datetime

# A tiny fake "document store" so search_documents has something to search.
DOCS = {
    "doc1": {"title": "Q3 Sales Report",
             "content": "Q3 revenue was 542000 dollars across 3 regions. North region contributed 210000 dollars."},
    "doc2": {"title": "Employee Handbook",
             "content": "Standard work week is 40 hours. Overtime rate is 1.5x base pay."},
    "doc3": {"title": "Project Alpha Budget",
             "content": "Project Alpha budget is 128000 dollars with 4 team members, each allocated an equal share."},
}

def search_documents(query: str, top_k: int = 1):
    """Search the internal document store and return the top matching documents."""
    query_l = query.lower()
    scored = []
    for doc_id, doc in DOCS.items():
        score = sum(1 for w in query_l.split() if w in doc["content"].lower() or w in doc["title"].lower())
        if score > 0:
            scored.append((score, doc_id, doc))
    scored.sort(key=lambda x: -x[0])
    results = [{"doc_id": d, "title": doc["title"], "content": doc["content"]} for _, d, doc in scored[:top_k]]
    return {"query": query, "results": results}

def get_weather_stub(city: str, unit: str = "celsius"):
    """Return a stubbed current-weather reading for a city."""
    fake_temps_c = {"delhi": 34, "new york": 21, "london": 17, "tokyo": 26}
    temp_c = fake_temps_c.get(city.lower(), 25)
    temp = temp_c * 9/5 + 32 if unit == "fahrenheit" else temp_c
    return {"city": city, "unit": unit, "temperature": temp, "condition": "clear"}

def calculate(expression: str):
    """Evaluate a basic arithmetic expression (+ - * / and parentheses only)."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        raise ValueError(f"Unsafe or invalid characters in expression: {expression!r}")
    try:
        result = eval(expression, {"__builtins__": {}}, {})
    except Exception as e:
        raise ValueError(f"Could not evaluate expression {expression!r}: {e}")
    return {"expression": expression, "result": result}

def get_today():
    """Return today's date in ISO 8601 format. Takes no arguments."""
    return {"date": datetime.date.today().isoformat()}

TOOLS = {
    "search_documents": search_documents,
    "get_weather_stub": get_weather_stub,
    "calculate": calculate,
    "get_today": get_today,
}

# sanity check
print(search_documents("Q3 revenue"))
print(get_weather_stub("Delhi", "fahrenheit"))
print(calculate("210000 / 3"))
print(get_today())


{'query': 'Q3 revenue', 'results': [{'doc_id': 'doc1', 'title': 'Q3 Sales Report', 'content': 'Q3 revenue was 542000 dollars across 3 regions. North region contributed 210000 dollars.'}]}
{'city': 'Delhi', 'unit': 'fahrenheit', 'temperature': 93.2, 'condition': 'clear'}
{'expression': '210000 / 3', 'result': 70000.0}
{'date': '2026-08-29'}


## 2. Day 26 recap — manual text-parsing agent

This mirrors the Day 26 approach: the "LLM" emits raw text like

```
Thought: I need the weather.
Action: get_weather_stub(city="Delhi", unit="fahrenheit")
```

and the agent uses a regex to pull out the tool name and arguments. It works
fine *as long as the model's formatting matches the regex exactly*. Below,
`DAY26_SCRIPT` plays the role of the LLM's raw text output for our five test
questions, and `DAY26_DRIFT_EXAMPLES` shows three realistic ways a model can
phrase the same intent that this parser fails on.

In [2]:
ACTION_RE = re.compile(r'Action:\s*(\w+)\((.*)\)', re.DOTALL)

def parse_day26_action(text):
    """Fragile regex parser -- mirrors a typical Day 26 manual ReAct parser."""
    m = ACTION_RE.search(text)
    if not m:
        return None
    tool_name, arg_str = m.group(1), m.group(2)
    args = {}
    for part in re.findall(r'(\w+)\s*=\s*"([^"]*)"|(\w+)\s*=\s*([^,]+)', arg_str):
        if part[0]:
            args[part[0]] = part[1]
        elif part[2]:
            args[part[2]] = part[3].strip()
    return tool_name, args

# Scripted raw LLM text for the 5 problems (+ their follow-up turns)
DAY26_SCRIPT = {
    "What's the weather in Delhi in Fahrenheit?":
        'Thought: I need the weather.\nAction: get_weather_stub(city="Delhi", unit="fahrenheit")',
    "What is today's date?":
        "Thought: easy.\nAction: get_today()",
    "How much is 12500 * 3?":
        'Thought: math.\nAction: calculate(expression="12500 * 3")',
    "Find the Q3 sales report and tell me the total revenue.":
        'Thought: search first.\nAction: search_documents(query="Q3 sales report revenue")',
    "What's the per-person budget share for Project Alpha?":
        'Thought: I should look this up.\nAction: search_documents(query="Project Alpha budget")',
    "__continue_alpha__":
        'Thought: budget is 128000 for 4 members.\nAction: calculate(expression="128000 / 4")',
    "__continue_q3__":            "Thought: revenue found.\nFinal Answer: Q3 revenue was 542000 dollars.",
    "__continue_alpha_final__":   "Thought: got it.\nFinal Answer: Each team member is allocated 32000 dollars.",
    "__continue_weather__":       "Thought: got the weather.\nFinal Answer: It's 93.2F and clear in Delhi.",
    "__continue_date__":          "Thought: got the date.\nFinal Answer: Today is 2026-08-29.",
    "__continue_calc__":          "Thought: done.\nFinal Answer: 12500 * 3 = 37500.",
}

NEXT_STEP = {
    "What's the weather in Delhi in Fahrenheit?": "__continue_weather__",
    "What is today's date?": "__continue_date__",
    "How much is 12500 * 3?": "__continue_calc__",
    "Find the Q3 sales report and tell me the total revenue.": "__continue_q3__",
    "What's the per-person budget share for Project Alpha?": "__continue_alpha__",
    "__continue_alpha__": "__continue_alpha_final__",
}

def run_day26_agent(question):
    """Simulated manual ReAct loop that parses raw text output."""
    trace = []
    step_key = question
    while True:
        raw = DAY26_SCRIPT[step_key]
        trace.append({"raw_llm_text": raw})
        if "Final Answer:" in raw:
            answer = raw.split("Final Answer:")[-1].strip()
            trace.append({"final_answer": answer})
            return answer, trace
        parsed = parse_day26_action(raw)
        if parsed is None:
            trace.append({"error": "PARSE_FAILURE: could not extract an Action from raw text"})
            return None, trace
        tool_name, args = parsed
        try:
            result = TOOLS[tool_name](**args)   # NOTE: args are all raw strings, no type coercion
        except TypeError as e:
            trace.append({"error": f"ARG_TYPE_ERROR calling {tool_name}: {e}"})
            return None, trace
        trace.append({"tool_call": tool_name, "args": args, "result": result})
        step_key = NEXT_STEP[step_key]


In [3]:
# Three realistic formatting variants that break the Day 26 regex parser
DAY26_DRIFT_EXAMPLES = [
    'I will call the tool get_weather_stub with city Delhi and unit fahrenheit.',  # no "Action:" keyword at all
    'Action -> calculate[expression: "210000 / 3"]',                              # wrong brackets / separator
    'Action: search_documents("Q3 sales report")',                                # positional arg, no key=value
]

print("Day 26 parser failures on realistic model drift:\n")
for text in DAY26_DRIFT_EXAMPLES:
    parsed = parse_day26_action(text)
    status = "PARSED OK" if parsed and parsed[1] else "FAILED (None or empty args)"
    print(f"  input: {text!r}\n  -> {parsed}   [{status}]\n")


Day 26 parser failures on realistic model drift:

  input: 'I will call the tool get_weather_stub with city Delhi and unit fahrenheit.'
  -> None   [FAILED (None or empty args)]

  input: 'Action -> calculate[expression: "210000 / 3"]'
  -> None   [FAILED (None or empty args)]

  input: 'Action: search_documents("Q3 sales report")'
  -> ('search_documents', {})   [FAILED (None or empty args)]



## 3. Day 27 — JSON function schemas

Each tool is described with OpenAI's function-calling schema format:
`name`, `description`, and a `parameters` JSON Schema object with typed
fields. This is what gets passed as `tools=[...]` to
`client.chat.completions.create(...)`.

In [4]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "search_documents",
            "description": "Search the internal document store and return the top matching documents with their content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Free-text search query."},
                    "top_k": {"type": "integer", "description": "Number of documents to return.", "default": 1},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather_stub",
            "description": "Get a (stubbed) current weather reading for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. 'Delhi'."},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "Temperature unit."},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a basic arithmetic expression using + - * / and parentheses.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "A basic arithmetic expression, e.g. '128000 / 4'."},
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_today",
            "description": "Return today's date in ISO 8601 format. Takes no arguments.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
]

print(json.dumps(TOOL_SCHEMAS, indent=2))


[
  {
    "type": "function",
    "function": {
      "name": "search_documents",
      "description": "Search the internal document store and return the top matching documents with their content.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Free-text search query."
          },
          "top_k": {
            "type": "integer",
            "description": "Number of documents to return.",
            "default": 1
          }
        },
        "required": [
          "query"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_weather_stub",
      "description": "Get a (stubbed) current weather reading for a city.",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "City name, e.g. 'Delhi'."
          },
          "unit": {
            "type

## 4. Structured tool-calling loop

The loop:
1. Send the conversation + `tools=TOOL_SCHEMAS` to the model.
2. If `response.choices[0].message.tool_calls` is present, execute each
   requested function with `json.loads(tool_call.function.arguments)`,
   append a `role="tool"` message with the result, and loop again.
3. Once the model responds with plain `content` and no `tool_calls`, that's
   the final answer.

`OpenAIClientAdapter` below uses the **real OpenAI API** if `OPENAI_API_KEY`
is set, otherwise falls back to a **deterministic mock** with the identical
response shape, so this notebook runs fully offline for grading/demo
purposes and also works unmodified against the live API.

In [5]:
import os

SYSTEM_PROMPT = (
    "You are a helpful assistant with access to tools. Use search_documents to look up "
    "internal documents, calculate for arithmetic, get_weather_stub for weather, and "
    "get_today for the current date. Call tools as needed, then give a concise final answer."
)

# ---- Deterministic mock backend (response shape matches the real API) ----
class MockToolCall:
    def __init__(self, call_id, name, arguments_dict):
        self.id = call_id
        self.type = "function"
        self.function = type("F", (), {"name": name, "arguments": json.dumps(arguments_dict)})()

class MockMessage:
    def __init__(self, content=None, tool_calls=None):
        self.content = content
        self.tool_calls = tool_calls
        self.role = "assistant"

class MockChoice:
    def __init__(self, message):
        self.message = message

class MockResponse:
    def __init__(self, message):
        self.choices = [MockChoice(message)]

# Scripted "model behavior" for our 5 test problems, turn by turn.
SCRIPT = {
    "What's the weather in Delhi in Fahrenheit?": [
        ("tool", "get_weather_stub", {"city": "Delhi", "unit": "fahrenheit"}),
        ("final", "It's 93.2F and clear in Delhi."),
    ],
    "What is today's date?": [
        ("tool", "get_today", {}),
        ("final", "Today is 2026-08-29."),
    ],
    "How much is 12500 * 3?": [
        ("tool", "calculate", {"expression": "12500 * 3"}),
        ("final", "12500 * 3 = 37500."),
    ],
    "Find the Q3 sales report and tell me the total revenue.": [
        ("tool", "search_documents", {"query": "Q3 sales report revenue", "top_k": 1}),
        ("final", "Q3 revenue was 542000 dollars."),
    ],
    "What's the per-person budget share for Project Alpha?": [
        ("tool", "search_documents", {"query": "Project Alpha budget", "top_k": 1}),
        ("tool", "calculate", {"expression": "128000 / 4"}),
        ("final", "Each of the 4 team members is allocated 32000 dollars."),
    ],
}

class MockOpenAIClient:
    """Deterministic stand-in for openai.OpenAI().chat.completions.create(...)"""
    def __init__(self, script):
        self.script = script

    def create(self, *, messages, tools, question, step_idx, **kw):
        step = self.script[question][step_idx]
        if step[0] == "final":
            return MockResponse(MockMessage(content=step[1], tool_calls=None))
        _, name, args = step
        call = MockToolCall(f"call_{step_idx}", name, args)
        return MockResponse(MockMessage(content=None, tool_calls=[call]))


class OpenAIClientAdapter:
    """Uses the real OpenAI client if OPENAI_API_KEY is set, else the mock."""
    def __init__(self, script):
        self.api_key = os.environ.get("OPENAI_API_KEY")
        self.live = False
        if self.api_key:
            try:
                from openai import OpenAI
                self._client = OpenAI(api_key=self.api_key)
                self.live = True
            except Exception as e:
                print(f"Could not initialize live OpenAI client ({e}); falling back to mock.")
        if not self.live:
            self._client = MockOpenAIClient(script)

    def create(self, messages, question=None, step_idx=None):
        if self.live:
            return self._client.chat.completions.create(
                model="gpt-4o-mini", messages=messages, tools=TOOL_SCHEMAS,
            )
        return self._client.create(messages=messages, tools=TOOL_SCHEMAS, question=question, step_idx=step_idx)


client = OpenAIClientAdapter(SCRIPT)
print("Using live OpenAI API" if client.live else "Using deterministic mock client (no OPENAI_API_KEY set)")


Using deterministic mock client (no OPENAI_API_KEY set)


In [6]:
def run_day27_agent(question, client, verbose=False):
    """Structured function-calling agent loop."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    trace = []
    step_idx = 0
    while True:
        resp = client.create(messages, question=question, step_idx=step_idx)
        msg = resp.choices[0].message
        if msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]})
            for tc in msg.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)  # structured JSON -- always parses
                trace.append({"tool_call": name, "raw_arguments_json": tc.function.arguments, "parsed_args": args})
                try:
                    result = execute_tool_safely(name, args)
                    trace.append({"tool_result": result})
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
                except ToolArgumentError as e:
                    trace.append({"validation_error": str(e)})
                    return None, trace
            step_idx += 1
        else:
            trace.append({"final_answer": msg.content})
            return msg.content, trace


## 5. Argument validation wrapper

Structured output guarantees *syntactically* valid JSON, but the model can
still choose *semantically* wrong values — an out-of-range enum, a missing
required field, a malformed expression. `validate_tool_call` catches these
before they ever reach the underlying Python function.

In [7]:
class ToolArgumentError(Exception):
    pass

VALIDATORS = {
    "get_weather_stub": {
        "required": ["city"],
        "enums": {"unit": ["celsius", "fahrenheit"]},
    },
    "calculate": {
        "required": ["expression"],
        "custom": lambda args: (
            None if args.get("expression", "").strip()
                     and args["expression"].strip()[-1] not in "+-*/(."
                     and set(args["expression"]) <= set("0123456789+-*/(). ")
            else "expression is empty, malformed, or contains disallowed characters"
        ),
    },
    "search_documents": {"required": ["query"]},
    "get_today": {"required": []},
}

def validate_tool_call(name, args):
    spec = VALIDATORS.get(name)
    if spec is None:
        raise ToolArgumentError(f"Unknown tool '{name}'")
    for field in spec.get("required", []):
        if field not in args or args[field] in (None, ""):
            raise ToolArgumentError(f"{name}: missing required argument '{field}'")
    for field, allowed in spec.get("enums", {}).items():
        if field in args and args[field] not in allowed:
            raise ToolArgumentError(f"{name}: '{field}'={args[field]!r} not in allowed values {allowed}")
    custom = spec.get("custom")
    if custom:
        err = custom(args)
        if err:
            raise ToolArgumentError(f"{name}: {err}")
    return True

def execute_tool_safely(name, args):
    validate_tool_call(name, args)
    return TOOLS[name](**args)


In [8]:
# Three cases where the model passes incorrect / missing arguments
BAD_ARG_CASES = [
    ("get_weather_stub", {"city": "Delhi", "unit": "kelvin"}),   # 1. invalid enum value (model hallucinated a unit)
    ("calculate", {"expression": "128000 / "}),                  # 2. malformed expression (missing operand)
    ("search_documents", {"top_k": 1}),                          # 3. missing required 'query' field
]

print("Validation wrapper catching bad tool-call arguments:\n")
for name, args in BAD_ARG_CASES:
    try:
        execute_tool_safely(name, args)
        print(f"  {name}({args}) -> unexpectedly passed validation")
    except ToolArgumentError as e:
        print(f"  {name}({args})\n    -> CAUGHT: {e}\n")


Validation wrapper catching bad tool-call arguments:

  get_weather_stub({'city': 'Delhi', 'unit': 'kelvin'})
    -> CAUGHT: get_weather_stub: 'unit'='kelvin' not in allowed values ['celsius', 'fahrenheit']

  calculate({'expression': '128000 / '})
    -> CAUGHT: calculate: expression is empty, malformed, or contains disallowed characters

  search_documents({'top_k': 1})
    -> CAUGHT: search_documents: missing required argument 'query'



## 6. Five-problem comparison: Day 26 vs Day 27

Same five multi-step questions used on Day 26, run through both agents.

In [9]:
QUESTIONS = [
    "What's the weather in Delhi in Fahrenheit?",
    "What is today's date?",
    "How much is 12500 * 3?",
    "Find the Q3 sales report and tell me the total revenue.",
    "What's the per-person budget share for Project Alpha?",
]

rows = []
for q in QUESTIONS:
    d26_answer, d26_trace = run_day26_agent(q)
    d27_answer, d27_trace = run_day27_agent(q, client)
    same_tools = (
        [t["tool_call"] for t in d26_trace if "tool_call" in t]
        == [t["tool_call"] for t in d27_trace if "tool_call" in t]
    )
    rows.append({
        "question": q,
        "day26_answer": d26_answer,
        "day27_answer": d27_answer,
        "same_tool_selection": same_tools,
        "same_final_answer": (d26_answer == d27_answer),
    })

for r in rows:
    print(f"Q: {r['question']}")
    print(f"  Day 26 -> {r['day26_answer']}")
    print(f"  Day 27 -> {r['day27_answer']}")
    print(f"  same tools selected: {r['same_tool_selection']}   same final answer: {r['same_final_answer']}\n")


Q: What's the weather in Delhi in Fahrenheit?
  Day 26 -> It's 93.2F and clear in Delhi.
  Day 27 -> It's 93.2F and clear in Delhi.
  same tools selected: True   same final answer: True

Q: What is today's date?
  Day 26 -> Today is 2026-08-29.
  Day 27 -> Today is 2026-08-29.
  same tools selected: True   same final answer: True

Q: How much is 12500 * 3?
  Day 26 -> 12500 * 3 = 37500.
  Day 27 -> 12500 * 3 = 37500.
  same tools selected: True   same final answer: True

Q: Find the Q3 sales report and tell me the total revenue.
  Day 26 -> Q3 revenue was 542000 dollars.
  Day 27 -> Q3 revenue was 542000 dollars.
  same tools selected: True   same final answer: True

Q: What's the per-person budget share for Project Alpha?
  Day 26 -> Each team member is allocated 32000 dollars.
  Day 27 -> Each of the 4 team members is allocated 32000 dollars.
  same tools selected: True   same final answer: False



In [10]:
import pandas as pd
pd.DataFrame(rows)[["question", "day26_answer", "day27_answer", "same_tool_selection", "same_final_answer"]]


,question,day26_answer,day27_answer,same_tool_selection,same_final_answer
0,What's the weather in Delhi in Fahrenheit?,It's 93.2F and clear in Delhi.,It's 93.2F and clear in Delhi.,True,True
1,What is today's date?,Today is 2026-08-29.,Today is 2026-08-29.,True,True
2,How much is 12500 * 3?,12500 * 3 = 37500.,12500 * 3 = 37500.,True,True
3,Find the Q3 sales report and tell me the total...,Q3 revenue was 542000 dollars.,Q3 revenue was 542000 dollars.,True,True
4,What's the per-person budget share for Project...,Each team member is allocated 32000 dollars.,Each of the 4 team members is allocated 32000 ...,True,False


## 7. Two-sequential-tool-call trace

The Project Alpha budget question requires **search_documents** (to find the
budget and headcount) followed by **calculate** (to divide budget by
headcount). Full exchange, both tool calls, shown below.

In [11]:
question = "What's the per-person budget share for Project Alpha?"
answer, trace = run_day27_agent(question, client)

print(f"Question: {question}\n")
for i, step in enumerate(trace):
    print(f"--- step {i} ---")
    print(json.dumps(step, indent=2, default=str))
print(f"\nFINAL ANSWER: {answer}")


Question: What's the per-person budget share for Project Alpha?

--- step 0 ---
{
  "tool_call": "search_documents",
  "raw_arguments_json": "{\"query\": \"Project Alpha budget\", \"top_k\": 1}",
  "parsed_args": {
    "query": "Project Alpha budget",
    "top_k": 1
  }
}
--- step 1 ---
{
  "tool_result": {
    "query": "Project Alpha budget",
    "results": [
      {
        "doc_id": "doc3",
        "title": "Project Alpha Budget",
        "content": "Project Alpha budget is 128000 dollars with 4 team members, each allocated an equal share."
      }
    ]
  }
}
--- step 2 ---
{
  "tool_call": "calculate",
  "raw_arguments_json": "{\"expression\": \"128000 / 4\"}",
  "parsed_args": {
    "expression": "128000 / 4"
  }
}
--- step 3 ---
{
  "tool_result": {
    "expression": "128000 / 4",
    "result": 32000.0
  }
}
--- step 4 ---
{
  "final_answer": "Each of the 4 team members is allocated 32000 dollars."
}

FINAL ANSWER: Each of the 4 team members is allocated 32000 dollars.


## 8. FastAPI `POST /agent` endpoint

Accepts `{"question": "..."}` and returns the final answer plus the complete
list of tool calls made during the interaction. Written to `agent_api.py` so
it can also be run standalone with:

```bash
uvicorn agent_api:app --reload
```

Below, the app is exercised in-notebook with FastAPI's `TestClient` (no
separate server process needed) to confirm it works.

In [12]:
%%writefile agent_api.py
"""
Day 27 FastAPI agent endpoint.
Run with:  uvicorn agent_api:app --reload
"""
import json
import os
from typing import Optional
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Day 27 Function-Calling Agent")


class AgentRequest(BaseModel):
    question: str


class AgentResponse(BaseModel):
    answer: Optional[str]
    tool_calls: list


@app.post("/agent", response_model=AgentResponse)
def agent_endpoint(req: AgentRequest):
    """
    Runs the structured function-calling agent loop for the given question
    and returns the final answer plus every tool call made along the way.

    NOTE: import the loop, client, and tool schemas defined in the notebook /
    a shared module (agent_core.py) when running this as a standalone service.
    """
    from agent_core import run_day27_agent, OpenAIClientAdapter, SCRIPT
    client = OpenAIClientAdapter(SCRIPT)
    answer, trace = run_day27_agent(req.question, client)
    tool_calls = [t for t in trace if "tool_call" in t]
    return AgentResponse(answer=answer, tool_calls=tool_calls)


Writing agent_api.py


In [13]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional

# In-notebook FastAPI app reusing the objects already defined above,
# so the /agent endpoint can be smoke-tested without a separate process.
demo_app = FastAPI(title="Day 27 Function-Calling Agent (in-notebook demo)")

class AgentRequest(BaseModel):
    question: str

class AgentResponse(BaseModel):
    answer: Optional[str]
    tool_calls: list

@demo_app.post("/agent", response_model=AgentResponse)
def agent_endpoint(req: AgentRequest):
    answer, trace = run_day27_agent(req.question, client)
    tool_calls = [t for t in trace if "tool_call" in t]
    return AgentResponse(answer=answer, tool_calls=tool_calls)

test_client = TestClient(demo_app)
resp = test_client.post("/agent", json={"question": "What's the per-person budget share for Project Alpha?"})
print("status:", resp.status_code)
print(json.dumps(resp.json(), indent=2))


status: 200
{
  "answer": "Each of the 4 team members is allocated 32000 dollars.",
  "tool_calls": [
    {
      "tool_call": "search_documents",
      "raw_arguments_json": "{\"query\": \"Project Alpha budget\", \"top_k\": 1}",
      "parsed_args": {
        "query": "Project Alpha budget",
        "top_k": 1
      }
    },
    {
      "tool_call": "calculate",
      "raw_arguments_json": "{\"expression\": \"128000 / 4\"}",
      "parsed_args": {
        "expression": "128000 / 4"
      }
    }
  ]
}


## 9. Reliability comparison document

### Day 26 (manual text parsing) vs Day 27 (structured function calling)

**1. Failure mode: parsing vs validation**
Day 26's regex parser (`Action:\s*(\w+)\((.*)\)`) only works if the model's
free-text output matches an exact shape. Section 2 above shows three
realistic — not contrived — variations that break it outright:
- `"I will call the tool get_weather_stub with city Delhi..."` → no `Action:`
  keyword at all → parser returns `None`, agent has no idea a tool call was
  even intended.
- `"Action -> calculate[expression: ...]"` → different arrow and bracket
  style → `None`.
- `'Action: search_documents("Q3 sales report")'` → positional argument
  instead of `key="value"` → the tool name is recovered but **`args` comes
  back empty**, so `search_documents()` gets called with no `query` and
  either errors or silently searches for nothing.

None of these are contrived — they're all things a real LLM does under
normal sampling variance (different phrasing conventions, occasional
free-form reasoning before committing to an action, one-off deviations from
a few-shot example). Because the parser only understands *one* shape of
text, the failure mode is binary: either it matches, or the agent silently
breaks with no signal about *why*.

Day 27's model output isn't free text to parse — `tool_calls[i].function.
arguments` is **guaranteed valid JSON** by the API contract itself. There's
no regex, no bracket-matching, no quote-handling. `json.loads()` either
succeeds (always, for a compliant API response) or the SDK itself raises
before your code even runs. This eliminates the entire class of "the model
phrased its intent slightly differently" failures.

**2. What structured calling does *not* fix**
Structured output guarantees *syntax*, not *semantics*. The model can still
emit a syntactically perfect tool call with the *wrong* values — Section 5's
three `BAD_ARG_CASES`:
- `get_weather_stub(unit="kelvin")` — valid JSON, invalid enum value.
- `calculate(expression="128000 / ")` — valid JSON, malformed arithmetic.
- `search_documents(top_k=1)` — valid JSON, but missing the required
  `query` field entirely.

This is why a **validation wrapper** (Section 5) is still necessary even
with function calling: it checks required fields, enum membership, and
custom per-tool invariants *before* the arguments reach the real Python
function, converting what would otherwise be a `TypeError`/`KeyError`
deep inside your tool logic into a clean, catchable `ToolArgumentError`
with a specific, actionable message.

**3. Tool selection and answer parity**
Section 6's five-problem run shows the model selects the **same tools in
the same order** and reaches the **same final answers** under both
approaches when its output happens to match the Day 26 parser's expected
format — the two approaches are equivalent in the *happy path*. The
reliability gap only shows up under formatting variance, which is exactly
where Day 26 has no fallback and Day 27 degrades gracefully into a
validation error instead of a silent parse failure.

**4. Multi-step (two-tool) reliability**
The Project Alpha question (Section 7) requires two sequential tool calls
where the second call's arguments (`128000 / 4`) depend on the *result* of
the first call (`search_documents` returning the budget and headcount from
document text). With Day 26, this dependency chain has to survive **two**
separate free-text parses in a row — doubling the surface area for a
formatting break. With Day 27, each step still goes through the same
guaranteed-JSON contract, so chaining tool calls doesn't compound the
failure risk the way it does with text parsing.

**5. Summary**

| Dimension | Day 26 (text parsing) | Day 27 (function calling) |
|---|---|---|
| Tool-name/argument extraction | Regex over free text — breaks on format drift | Structured JSON — guaranteed parseable |
| Failure visibility | Silent `None`/empty-args, easy to miss | Explicit exception at a known point in the loop |
| Semantic argument errors (wrong enum, missing field) | Also possible, and *harder* to detect since it's mixed with parse errors | Isolated to a single, testable validation layer |
| Multi-step chains | Failure risk compounds with each additional free-text parse | Failure risk stays flat — each step uses the same guaranteed contract |
| Effort to harden | Must anticipate every text format the model might drift into | Must anticipate every semantically-wrong-but-valid value the model might send |

**Bottom line:** function calling doesn't make the model *smarter* about
picking correct arguments — the "wrong Kelvin unit" case proves it can still
be wrong. What it removes is an entire, unbounded category of *formatting*
failures, replacing "did the model's free text happen to match my regex"
with "does the model's chosen value pass my validator" — a much smaller,
much more testable surface.
